<a href="https://colab.research.google.com/github/dkalchenko01/UA-Locations-NER-Deep-Learning/blob/quality/notebooks/07_reannotations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reannotation experiment

Comparison of four annotation strategies using the same
BiLSTM architecture, tokenizer and training configuration.

Evaluation datasets:
- Validation v2
- Labeling sample

## Setup

In [1]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "main",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)

/content/UA-Locations-NER-Deep-Learning


In [2]:
%pip install -q -r "$repo_dir/requirements.txt"

In [3]:
import gc
import json
import ast
import pandas as pd
import torch

from time import perf_counter
from tokenizers import Tokenizer
from google.colab import drive

from src.data_io import get_data_dir, load_manifest, load_data_splits, save_json, save_predictions, file_sha256
from src.bigru_helpers import CLEAN_OPTIONS, postprocessed_scores
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader, LABEL_TO_ID
from src.model import BiLSTMTagger
from src.metrics import competition_scores
from src.postprocessing import postprocess
from src.training import set_seed, train_epoch, predict, save_checkpoint

In [4]:
drive.mount("/content/gdrive")

Mounted at /content/gdrive


## Configuration

Using the same BiLSTM architecture and training parameters
for all annotation variants.

In [5]:
config = {
    "model": "bilstm", "architecture": "stacked", "embedding_dim": 64, "hidden_size": 64,
    "num_layers": 1, "dropout": 0.2, "seed": 17, "batch_size": 16,
    "learning_rate": 1e-3, "epochs": 15, "gradient_clip": 1.0,
    "max_padded_tokens": 16384, "prepare_batch_size": 2000,
    "sources": ["uk"], "optimizer": "Adam",
    "tokenizer": "bpe_whitespace.json", "selection_metric": "uk_comp_f1",
    "label_to_id": LABEL_TO_ID, "max_sequence_length": None,
    "early_stopping": False, "environment": ENVIRONMENT,
    "validation_version": "v2", "manifest_name": "validation_manifest_v2",
    "metric_version": "kaggle_original_regex_v1",
    "postprocessing_version": "expand_trim_empty", "eval_every": 1,
    "decoding": "viterbi_lines", "clean_options": dict(CLEAN_OPTIONS)
}

In [20]:
DATASETS_TO_RUN = ["clean_reannotation"]
FORCE_RERUN = False
SEEDS = [17, 42, 123]

ANNOTATIONS_DIR = Path("/content/gdrive/MyDrive/ner_results/llm annotations")
RESULTS_DIR = ANNOTATIONS_DIR / "experiment_results_v3"

DATASET_PATHS = {
    "original": ANNOTATIONS_DIR / "experiment_sample_1000.csv",
    "clean_reannotation": (
        ANNOTATIONS_DIR
        / "clean_reannotation_v3"
        / "annotated_sample_clean.csv"
    ),
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
set_seed(config["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

splits_dir = repo_dir / "data/splits"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]
manifest_path = splits_dir / f"{config['manifest_name']}.parquet"

print("Device:", device)
print("Datasets to run:", DATASETS_TO_RUN)

Device: cuda
Datasets to run: ['clean_reannotation']


## Data preparation

Reusing the saved tokenizer and validation v2 splits.
All annotation variants use the same evaluation datasets.

In [9]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
pad_id = tokenizer.token_to_id("[PAD]")

data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir, name=config["manifest_name"])

display(pd.crosstab(manifest["source"], manifest["split"]))

100%|██████████| 811M/811M [00:11<00:00, 72.1MB/s]

Extracting files...


split,holdout,train,val
source,,,
ru,24743,100000,24626
uk,100757,805983,100741


validation

In [10]:
splits = load_data_splits(ENVIRONMENT, splits_dir, sources=("uk",), manifest_name=config["manifest_name"])
val_df = splits["uk_val"]

del splits
gc.collect()

print("Validation:", len(val_df))

Validation: 100741


In [11]:
val_records, val_skipped = prepare_ner_records(
    tokenizer, val_df, "uk", "val", batch_size=config["prepare_batch_size"]
)
print("Validation records:", len(val_records))

Prepare uk/val:   0%|          | 0/51 [00:00<?, ?it/s]

Validation records: 100741


In [12]:
loader_options = {
    "pad_id": pad_id, "batch_size": config["batch_size"], "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"], "pin_memory": device.type == "cuda"
}

eval_loader_options = {**loader_options, "batch_size": 128}
val_loader = make_loader(val_records, **eval_loader_options)

`labeling_sample.csv`

In [13]:
labeling_df = pd.read_csv(data_dir / "labeling_sample.csv")

print("Labeling sample:", len(labeling_df))
print("Columns:", labeling_df.columns.tolist())
display(labeling_df.head(3))

Labeling sample: 26
Columns: ['text', 'locations', 'loc_markers']


,text,locations,loc_markers
0,-261.310,[],[]
1,"Проспект Бандери стоїть, як і пів міста… довго...",['Проспект Бандери'],"[(0, 16)]"
2,☢️Радіаційний фон (мкР/год) по Київщині станом...,"['Київщині', 'Чорнобиль', 'Київ', 'Тетерів', '...","[(31, 39), (68, 77), (83, 87), (93, 100), (105..."


In [14]:
labeling_records, labeling_skipped = prepare_ner_records(
    tokenizer, labeling_df, "uk", "labeling_sample", batch_size=config["prepare_batch_size"]
)

labeling_loader = make_loader(labeling_records, **eval_loader_options)
print("Labeling sample records:", len(labeling_records))
print("Alignment failures:", len(labeling_skipped))

Prepare uk/labeling_sample:   0%|          | 0/1 [00:00<?, ?it/s]

Labeling sample records: 26
Alignment failures: 1


## Training datasets

In [15]:
original_df = pd.read_csv(DATASET_PATHS["original"]).set_index("row_id").sort_index()

train_records_by_dataset = {}
alignment = []

for name in DATASETS_TO_RUN:
    if not FORCE_RERUN and all((RESULTS_DIR / name / f"seed_{seed}" / "metrics.json").exists() for seed in SEEDS):
        print(f"Skipping completed dataset: {name}")
        continue

    train_df = pd.read_csv(DATASET_PATHS[name]).set_index("row_id").sort_index()
    records, skipped = prepare_ner_records(tokenizer, train_df, "uk", "train", batch_size=config["prepare_batch_size"])
    train_records_by_dataset[name] = records
    alignment.append({
        "dataset": name, "total": len(train_df),
        "retained": len(records), "alignment_failures": len(skipped)
    })

display(pd.DataFrame(alignment))

Prepare uk/train:   0%|          | 0/1 [00:00<?, ?it/s]

,dataset,total,retained,alignment_failures
0,clean_reannotation,1000,999,1


## Model

The same stacked BiLSTM is initialized from scratch
for each annotation variant.

In [16]:
model_config = {
    "vocab_size": tokenizer.get_vocab_size(), "pad_id": pad_id,
    "embedding_dim": config["embedding_dim"], "hidden_size": config["hidden_size"],
    "num_layers": config["num_layers"], "dropout": config["dropout"],
    "num_labels": len(LABEL_TO_ID)
}

## Training



In [17]:
def run_experiment(name, train_records, seed):
    run_dir = RESULTS_DIR / name / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    run_config = {**config, "seed": seed}

    set_seed(seed)
    train_loader = make_loader(train_records, shuffle=True, **{**loader_options, "seed": seed})

    model = BiLSTMTagger(**model_config).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])

    provenance = {
        "dataset": name,
        "dataset_sha256": file_sha256(DATASET_PATHS[name]),
        "tokenizer_sha256": file_sha256(tokenizer_path),
        "manifest_sha256": file_sha256(manifest_path),
        "git_revision": subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True).strip()
    }

    history = []

    for epoch in range(1, config["epochs"] + 1):
        print(f"\n{name} | Epoch {epoch}/{config['epochs']}")
        epoch_started = perf_counter()

        train_loader.batch_sampler.set_epoch(epoch)
        train_loss = train_epoch(model, train_loader, optimizer, device, config["gradient_clip"])

        if epoch % config["eval_every"] != 0 and epoch != config["epochs"]:
          print(f"Train loss: {train_loss:.6f}")
          continue

        val_predictions, val_losses = predict(model, val_loader, device, decoding=config["decoding"])
        val_scores = postprocessed_scores(val_records, val_predictions, config["clean_options"])

        labeling_predictions, labeling_losses = predict(model, labeling_loader, device, decoding=config["decoding"])
        labeling_scores = postprocessed_scores(labeling_records, labeling_predictions, config["clean_options"])

        result = {
            "epoch": epoch, "train_loss": train_loss,
            "val_loss": val_losses["uk"], "labeling_loss": labeling_losses["uk"],
            **{f"val_{k}": v for k, v in val_scores.items()},
            **{f"labeling_{k}": v for k, v in labeling_scores.items()},
            "epoch_seconds": perf_counter() - epoch_started
        }

        history.append(result)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        save_checkpoint(run_dir / f"epoch_{epoch:02d}.pt", model, optimizer, epoch, run_config, model_config, result, provenance)
        print(result)
        del val_predictions, labeling_predictions

    summary = {
      "dataset": name,
      "seed": seed,
      "train_size": len(train_records),
      "best_f1_epoch": max(history, key=lambda r: r["val_uk_comp_f1"])["epoch"],
      "best_loss_epoch": min(history, key=lambda r: r["val_loss"])["epoch"]
    }

    save_json(run_dir / "config.json", run_config)
    save_json(run_dir / "provenance.json", provenance)
    save_json(run_dir / "metrics.json", summary)

    del model, optimizer
    gc.collect()
    torch.cuda.empty_cache()

    return summary

## Run experiments

Train only the selected annotation variants.
Previously completed experiments are skipped unless FORCE_RERUN is enabled.

In [18]:
for name, records in train_records_by_dataset.items():
  for seed in SEEDS:
    if not FORCE_RERUN and (RESULTS_DIR / name / f"seed_{seed}" / "metrics.json").exists():
        print(f"Skipping completed experiment: {name}, seed {seed}")
        continue

    summary = run_experiment(name, records, seed)
    print(summary)


clean_reannotation | Epoch 1/15


Train:   0%|          | 0/63 [00:00<?, ?it/s]

Predict:   0%|          | 0/788 [00:00<?, ?it/s]

Predict:   0%|          | 0/1 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.4584674675659396, 'val_loss': 0.15165775202738202, 'labeling_loss': 0.21343786668861464, 'val_uk_comp_f1': 0, 'val_combined_comp_f1': 0, 'val_precision': 0.0, 'val_recall': 0.0, 'val_TP': 0, 'val_FP': 0, 'val_FN': 34005, 'labeling_uk_comp_f1': 0, 'labeling_combined_comp_f1': 0, 'labeling_precision': 0.0, 'labeling_recall': 0.0, 'labeling_TP': 0, 'labeling_FP': 0, 'labeling_FN': 32, 'epoch_seconds': 22.283495021999954}

clean_reannotation | Epoch 2/15


Train:   0%|          | 0/63 [00:00<?, ?it/s]

... (171 outputs left out)

Train:   0%|          | 0/63 [00:00<?, ?it/s]

Predict:   0%|          | 0/788 [00:00<?, ?it/s]

Predict:   0%|          | 0/1 [00:00<?, ?it/s]

{'epoch': 15, 'train_loss': 0.02053311595808588, 'val_loss': 0.2021257440481312, 'labeling_loss': 0.34802113422665676, 'val_uk_comp_f1': 0.23337989, 'val_combined_comp_f1': 0.23337989, 'val_precision': 0.3045351259231206, 'val_recall': 0.1891780620496986, 'val_TP': 6433, 'val_FP': 14691, 'val_FN': 27572, 'labeling_uk_comp_f1': 0.08510638, 'labeling_combined_comp_f1': 0.08510638, 'labeling_precision': 0.13333333333333333, 'labeling_recall': 0.0625, 'labeling_TP': 2, 'labeling_FP': 13, 'labeling_FN': 30, 'epoch_seconds': 22.412134802000082}
{'dataset': 'clean_reannotation', 'seed': 123, 'train_size': 999, 'best_f1_epoch': 14, 'best_loss_epoch': 6}


## Inspect results

In [22]:
from torch.nn.utils.rnn import pad_sequence
from src.dataset import ID_TO_LABEL
from src.decoding import viterbi_decode
from src.preprocessing import bio_to_spans

dataset_names = [
    "original",
    "guided_reannotation",
    "clean_reannotation",
    "cleaned_labels",
]
seeds = [17, 42, 123]

# Обираємо checkpoint за labeling_sample, потім за VAL
checkpoint_paths = {}
for name in dataset_names:
    checkpoint_paths[name] = []

    for seed in seeds:
        run_dir = RESULTS_DIR / name / f"seed_{seed}"
        history = pd.read_csv(run_dir / "history.csv")
        best = history.sort_values(
            ["labeling_uk_comp_f1", "val_uk_comp_f1", "epoch"],
            ascending=[False, False, True],
        ).iloc[0]

        epoch = int(best["epoch"])
        path = run_dir / f"epoch_{epoch:02d}.pt"
        if not path.is_file():
            raise FileNotFoundError(path)

        checkpoint_paths[name].append(path)
        print(
            f"{name} | seed {seed} | epoch {epoch} | "
            f"sample F1 {best['labeling_uk_comp_f1']:.5f} | "
            f"VAL F1 {best['val_uk_comp_f1']:.5f}"
        )

test_df = pd.read_csv(data_dir / "test.csv")
test_records = prepare_inference_records(tokenizer, test_df)
test_loader = make_loader(test_records, **eval_loader_options)

submission_dir = RESULTS_DIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)
tokenizer_hash = file_sha256(tokenizer_path)

for name, checkpoint_files in checkpoint_paths.items():
    probabilities = {}
    cleanup = None

    for checkpoint_path in checkpoint_files:
        checkpoint = torch.load(
            checkpoint_path, map_location="cpu", weights_only=False
        )
        run_config = checkpoint["config"]

        if checkpoint["provenance"]["tokenizer_sha256"] != tokenizer_hash:
            raise ValueError(f"Different tokenizer: {checkpoint_path}")
        if cleanup is not None and cleanup != run_config["clean_options"]:
            raise ValueError(f"Different cleanup settings: {checkpoint_path}")
        cleanup = run_config["clean_options"]

        model = BiLSTMTagger(**checkpoint["model_config"]).to(device)
        model.load_state_dict(checkpoint["model_state_dict"])

        predictions, _ = predict(
            model, test_loader, device, return_logits=True,
        )

        for record in test_records:
            text_id = record["text_id"]
            n_tokens = len(record["input_ids"])
            probs = torch.tensor(
                predictions[text_id]["logits"], dtype=torch.float32
            ).reshape(n_tokens, len(ID_TO_LABEL)).softmax(dim=-1)

            if text_id not in probabilities:
                probabilities[text_id] = probs / len(checkpoint_files)
            else:
                probabilities[text_id] += probs / len(checkpoint_files)

        del model, checkpoint, predictions
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # Viterbi з обмеженнями переносів рядків
    locations = {record["text_id"]: [] for record in test_records}
    nonempty = [r for r in test_records if len(r["input_ids"])]

    for start in range(0, len(nonempty), 128):
        batch = nonempty[start:start + 128]
        emissions = pad_sequence(
            [
                probabilities[r["text_id"]].clamp_min(1e-30).log()
                for r in batch
            ],
            batch_first=True,
        )
        tag_paths = viterbi_decode(
            emissions,
            [len(r["input_ids"]) for r in batch],
            records=batch,
        )

        for record, tags in zip(batch, tag_paths):
            spans = bio_to_spans(
                [ID_TO_LABEL[tag] for tag in tags],
                record["offsets"].tolist(),
            )
            locations[record["text_id"]] = postprocess(
                record["text"], spans, **cleanup,
            )

    submission = test_df[["text_id"]].copy()
    submission["locations"] = [
        json.dumps(locations[text_id], ensure_ascii=False)
        for text_id in submission["text_id"]
    ]

    path = submission_dir / f"submission_{name}_3seeds_sample_selected.csv"
    submission.to_csv(path, index=False)
    print("Saved:", path)

    del probabilities
    gc.collect()

original | seed 17 | epoch 15 | sample F1 0.04762 | VAL F1 0.25364
original | seed 42 | epoch 12 | sample F1 0.09524 | VAL F1 0.21610
original | seed 123 | epoch 13 | sample F1 0.12500 | VAL F1 0.23559
guided_reannotation | seed 17 | epoch 10 | sample F1 0.09756 | VAL F1 0.23036
guided_reannotation | seed 42 | epoch 9 | sample F1 0.08511 | VAL F1 0.18888
guided_reannotation | seed 123 | epoch 13 | sample F1 0.11538 | VAL F1 0.22820
clean_reannotation | seed 17 | epoch 13 | sample F1 0.09302 | VAL F1 0.24655
clean_reannotation | seed 42 | epoch 12 | sample F1 0.08889 | VAL F1 0.20472
clean_reannotation | seed 123 | epoch 14 | sample F1 0.08696 | VAL F1 0.23350
cleaned_labels | seed 17 | epoch 14 | sample F1 0.05405 | VAL F1 0.21008
cleaned_labels | seed 42 | epoch 14 | sample F1 0.05882 | VAL F1 0.16683
cleaned_labels | seed 123 | epoch 14 | sample F1 0.10811 | VAL F1 0.19600


Predict:   0%|          | 0/6 [00:00<?, ?it/s]

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

Saved: /content/gdrive/MyDrive/ner_results/llm annotations/experiment_results_v3/submissions/submission_original_3seeds_sample_selected.csv


Predict:   0%|          | 0/6 [00:00<?, ?it/s]

... (7 outputs left out)

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

Saved: /content/gdrive/MyDrive/ner_results/llm annotations/experiment_results_v3/submissions/submission_cleaned_labels_3seeds_sample_selected.csv
